In [1]:
import os
# Get the name of the current conda environment
env_name = os.getenv("CONDA_DEFAULT_ENV")
print(f"The current Conda environment is: {env_name}")

The current Conda environment is: tensorflowgpu


In [2]:
# import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
# import matplotlib.pyplot as plt
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Dense, Dropout, LSTM
from tensorflow.keras.layers import Layer, Multiply, Dense, Input
from tensorflow.keras.models import Model
from scipy.stats import pearsonr
import tensorflow as tf

# Check if TensorFlow can access a GPU
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [5]:
import numpy
print(numpy.__version__)

2.0.2


In [3]:
class AttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)
        self.attention_dense = None  # Will be initialized in build()

    def build(self, input_shape):
        self.attention_dense = Dense(input_shape[-1], activation='softmax')
        super(AttentionLayer, self).build(input_shape)

    def call(self, inputs):
        attention_weights = self.attention_dense(inputs)
        weighted_output = Multiply()([inputs, attention_weights])
        return weighted_output

    def compute_output_shape(self, input_shape):
        return input_shape

In [4]:
def nse(y_true, y_pred):
    return 1 - (np.sum((y_true - y_pred) ** 2) / np.sum((y_true - np.mean(y_true)) ** 2))
import numpy as np

def pbias(y_true, y_pred):
    pbias_value = 100 * np.sum(y_true - y_pred) / np.sum(y_true)
    return float(pbias_value)


def kge(y_true, y_pred):
    # Calculate the Pearson correlation coefficient (r)
    r, _ = pearsonr(y_true, y_pred)
    
    # Calculate the mean of the observed and predicted values
    mu_true = np.mean(y_true)
    mu_pred = np.mean(y_pred)
    
    # Calculate the standard deviation of the observed and predicted values
    sigma_true = np.std(y_true)
    sigma_pred = np.std(y_pred)
    
    # Compute the KGE
    kge_value = 1 - np.sqrt((r - 1)**2 + (sigma_pred / sigma_true - 1)**2 + (mu_pred / mu_true - 1)**2)
    
    return kge_value


In [5]:
# Define start and end dates as datetime objects
start_date = pd.to_datetime("2000-01-01")
end_date = pd.to_datetime("2100-12-31")

In [6]:
epochs=256
batch_size=16
os.chdir('F:/geodata/river_runoff_obs')
GCM_name = 'MPI-ESM1-2-HR'
['EC-Earth3',
 'FGOALS-g3',
'multi-model',
'BCC-CSM2-MR',
'MRI-ESM2-0',
'INM-CM5-0',
'INM-CM4-8']

['EC-Earth3',
 'FGOALS-g3',
 'multi-model',
 'BCC-CSM2-MR',
 'MRI-ESM2-0',
 'INM-CM5-0',
 'INM-CM4-8']

In [7]:
valid_rate = 0.3
input_file_name_list = ['1_hsg_imputMF','2_dsk_imputMF','3_xhl_imputMF','4_slglk_imputMF','5_kq_imputMF','6_wlwt_imputMF','7_tgzlk_imputMF']

In [8]:
from datetime import datetime# Get current time
current_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
# 创建文件名
txtbook_path = f'F:/geodata/river_runoff_obs/note/note_{current_time}_mon.txt'
print(txtbook_path)
for GCM_name in ['FGOALS-g3','MPI-ESM1-2-HR', 'EC-Earth3',  'BCC-CSM2-MR', 'MRI-ESM2-0', 'INM-CM5-0', 'INM-CM4-8']:
    print(GCM_name,epochs,batch_size,valid_rate)
    for input_file_name in input_file_name_list:
        print(epochs,batch_size,valid_rate)
        abbre = input_file_name.split('_')[1]
        for scenario in ['ssp126','ssp245','ssp370','ssp585']:
            print(scenario)
            
            # Dataset loading
            name = f'{input_file_name}_{GCM_name}_{scenario}_r1i1p1f1_mon'
            csv_path = f"{name}.csv"
            usecols = ['time', 'pre', 'tm','gr','dis','ep']
            df_full = pd.read_csv(csv_path, usecols=usecols)
            df_full = df_full[-1212:]
            df = df_full.dropna()
            df_future = df_full[df_full.dis.isnull()]
            file_name = f'{name}_{epochs}_{batch_size}'
            # Prepare features (X) and targets (y)
            X = df[['pre', 'tm','gr' ]].values  # Inputs: precipitation, temperature, mass balance
            y = df[['dis', 'ep']].values         # Outputs: runoff (dis) and evaporation (ep)
            
            # Split the data into training and testing sets
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=valid_rate, random_state=42)
            
            # Standardize the data
            scaler_X = StandardScaler()
            scaler_y = StandardScaler()
            
            X_train = scaler_X.fit_transform(X_train)
            X_test = scaler_X.transform(X_test)
            y_train = scaler_y.fit_transform(y_train)
            y_test = scaler_y.transform(y_test)     
    
    
    
            # Define the model
            inputs = Input(shape=(X_train.shape[1],))  # Input layer
            x = Dense(64, activation='relu')(inputs)  # Hidden layer 1
            x = Dense(32, activation='relu')(x)       # Hidden layer 2
            x = AttentionLayer()(x)                   # Attention layer
            x = Dense(2)(x)                           # Output layer (2 units for runoff and evaporation)
            
            # Create the model
            model = Model(inputs, x)
            
            # Add Dropout after defining the layers
            # model.add(Dropout(0.2))  # Apply a dropout rate of 20%
            
            # Compile the model
            model.compile(optimizer='adam', loss='mse')
            
            # Print the model summary
            model.summary()
            
            # Train the model
            model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=valid_rate, )
            
            # Make predictions
            predictions = model.predict(X_test)
            predictions_rescaled = scaler_y.inverse_transform(predictions)
            
            # Convert predictions to DataFrame for runoff and evaporation
            pred_df = pd.DataFrame(predictions_rescaled, columns=['predicted_runoff', 'predicted_ep'])
            print(pred_df.head())
            from sklearn.metrics import mean_squared_error
            # Calculate predictions and rescale
            predictions = model.predict(X_test)
            predictions_rescaled = scaler_y.inverse_transform(predictions)
            
            # Separate the predicted and actual values for runoff and evaporation
            y_test_rescaled = scaler_y.inverse_transform(y_test)
            runoff_observed = y_test_rescaled[:, 0]
            evaporation_observed = y_test_rescaled[:, 1]
            runoff_predicted = predictions_rescaled[:, 0]
            evaporation_predicted = predictions_rescaled[:, 1]
            
            
            # Assuming future_df is loaded and has the same columns as df
            # Extract features from future_df
            X_full = df_full[['pre', 'tm','gr' ]].values  # Only the input features
            # Standardize features based on the training data scaler
            X_full_scaled = scaler_X.transform(X_full)
            
            # Convert the 'time' column to datetime format if needed
            df.loc[:,'time'] = pd.to_datetime(df['time'])
            df_full.loc[:,'time'] = pd.to_datetime(df_full['time'])
            # Ensure 'time' is set as the index for both DataFrames if not already
            df.set_index('time', inplace=True)
            df_full.set_index('time', inplace=True)
            
    
            # Predict future runoff and evaporation
            full_predictions_scaled = model.predict(X_full_scaled)
            full_predictions_scaled = model.predict(X_full_scaled)
            
            # Rescale predictions to original scale
            full_predictions = scaler_y.inverse_transform(full_predictions_scaled)
            
            # Convert predictions to DataFrame for readability
            df_full[['Projected_Runoff', 'Projected_Evaporation']] = full_predictions
            print(df_full[['Projected_Runoff', 'Projected_Evaporation']].head())
            
    
    
            historic_pred_df = df_full.dropna(subset='dis')
            historic_pred_df = historic_pred_df.tail(int(valid_rate * len(historic_pred_df)))
            # Define the NSE function
        
            # Calculate NSE for runoff and evaporation
            nse_runoff = nse(historic_pred_df.dis, historic_pred_df.Projected_Runoff)
            nse_evaporation = nse(historic_pred_df.ep, historic_pred_df.Projected_Evaporation)
            kge_runoff = kge(historic_pred_df.dis, historic_pred_df.Projected_Runoff)
            pbias_runoff = pbias(historic_pred_df.dis, historic_pred_df.Projected_Runoff)
            print('nse_runoff:',nse_runoff,'nse_ep:',nse_evaporation,'kge_runoff:',kge_runoff,'pbias_runoff:',pbias_runoff)
            
        
            # # Plot the line chart for Projected_Runoff
            # plt.plot(historic_pred_df.index, historic_pred_df['Projected_Runoff'], label='Projected Runoff', color='red')
            #
            # # Plot the scatter plot for dis
            # plt.scatter(historic_pred_df.index, historic_pred_df['dis'], label='Observed Runoff', color='b')
            #
            # # Add labels, title, and legend
            # plt.xlabel('Index')
            # plt.ylabel(r'Runoff ($\mathrm{m^3 \cdot s^{-1}}$)')  # Use LaTeX for units
            # plt.title('Projected Runoff vs Observed Runoff')
            # plt.xlim(([pd.to_datetime("2014-01"), pd.to_datetime("2020-01-01")]))
            # plt.legend()
            #
            # # Show the plot
            # plt.show()
            #
            # # Plot with a larger figure size
            # ax = df_full[[ 'Projected_Runoff','dis']].plot(figsize=(12, 6))
            # # Optional: Rotate x-tick labels for better readability
            # plt.xticks(rotation=45)
            # plt.title(f"Historical and projected runoff of {file_name}")
            # # Show the plot
            # plt.show()
            # # Plot with a larger figure size
            # ax = df_full[['ep', 'Projected_Evaporation']].plot(figsize=(12, 6))
            # # Optional: Rotate x-tick labels for better readability
            # plt.xticks(rotation=45)
            # plt.title(f"Historical and projected evaporation of {file_name}")
            # # Show the plot
            # plt.show()
            #
            # # Generate a range of ticks every 20 years
            # x_ticks = pd.date_range(start=start_date, end=end_date, freq='10Y')
            #
            # # Set the size of the figure
            # fig, axes = plt.subplots(5, 1, figsize=(16, 9), sharex=True,facecolor='w')
            # title_list = ['Monthly Precipitation','Monthly Temperature','Monthly Glacier Runoff','Monthly Runoff','Monthly Evaporation']
            #
            # # Define y-axis limits for each plot
            # y_lims = [(0, 70), (-20, 25), (0, 3.3* 1e9), (0, 1200), (0, 800)]
            #
            # # Loop through the columns and set y-axis limits
            # for i, col in enumerate(['pre', 'tm','gr',  ['dis', 'Projected_Runoff'], ['ep', 'Projected_Evaporation']]):
            #     df_full[col].plot(ax=axes[i])
            #
            #     # # Set y-axis limits
            #     # axes[i].set_ylim(y_lims[i])
            #
            #     # Optional: Add y-axis label, legend, title, grid, etc.
            #     # axes[i].set_ylabel(col)  # Set y-axis label to the column name
            #     axes[i].legend(loc='upper right')  # Optional: add legend
            #     axes[i].set_title(title_list[i])
            #
            #     # Set x-axis limits
            #     axes[i].set_xlim([start_date, end_date])
            #     axes[i].set_xticks(x_ticks)
            #
            #     # Set x-axis labels as years (2000, 2020, ..., 2100)
            #     axes[i].set_xticklabels([str(date.year) for date in x_ticks], rotation=0)
            #
            #     axes[i].grid(True)  # Optional: add grid for clarity
            #
            #
            #
            #
            # # Set x-axis label for the entire figure
            # axes[-1].set_xlabel('Date')  # or adjust label based on your x-axis
            #
            # # Adjust layout to prevent overlap
            # plt.tight_layout()
            # plt.savefig(f'{file_name}.svg')
            # plt.show()
            
            '''df_full.to_csv(f"{file_name}_project.csv")'''
            # annual_df = df_full.resample('Y').agg({
            #     'pre': 'sum',  # If there are any NaNs in the group, the sum will be NaN
            #     'tm': 'mean',  # The mean will also return NaN if there are NaNs in the group
            #     'gr': 'sum',   # Same for sum, will return NaN if any NaNs are present
            #     'dis': 'sum',
            #     'ep': 'sum',
            #     'Projected_Runoff': 'sum',
            #     'Projected_Evaporation': 'sum'
            # }, skipna=False)  # Ensures NaNs are preserved in aggregation
            #
            # # Reset index if needed
            # annual_df.reset_index(inplace=True)
            # annual_df[['ep', 'dis', 'Projected_Runoff',  'Projected_Evaporation']]=annual_df[['ep', 'dis', 'Projected_Runoff',  'Projected_Evaporation']].replace(0, np.nan,)
            #
            # # Display the result
            # print(annual_df.head())

            # --- Prepare input sample ---
            # Select one instance to explain (e.g., the first sample in test set)
            input_index = 0
            X_sample = X_test[input_index]  # already standardized
            baseline = np.zeros_like(X_sample)  # baseline (zero vector)
            baseline = baseline.astype(np.float32)
            X_sample = X_sample.astype(np.float32)

            # --- Define IG function ---
            @tf.function
            def interpolate_inputs(baseline, input, alphas):
                return baseline + alphas * (input - baseline)

            def compute_integrated_gradients(model, baseline, input, steps=100):
                alphas = tf.linspace(0.0, 1.0, steps + 1)
                interpolated_inputs = interpolate_inputs(baseline, input, alphas[:, tf.newaxis])
                interpolated_inputs = tf.convert_to_tensor(interpolated_inputs)

                with tf.GradientTape() as tape:
                    tape.watch(interpolated_inputs)
                    predictions = model(interpolated_inputs)
                    # Target output index: 0 for runoff ('dis'), 1 for evaporation ('ep')
                    target = predictions[:, 1]  # Change to predictions[:, 1] for evaporation

                grads = tape.gradient(target, interpolated_inputs)
                avg_grads = tf.reduce_mean(grads, axis=0)
                integrated_grads = (input - baseline) * avg_grads

                return integrated_grads.numpy()

            # --- Run IG ---
            ig = compute_integrated_gradients(model, baseline, X_sample)
            print(ig)

            # append the information at the end of text book
            with open(txtbook_path, 'a') as file:
                file.write(f"Name: {name}, Scenario: {scenario}, Epochs: {epochs}, Batch Size: {batch_size}, Valid Rate: {valid_rate},NSE Runoff: {nse_runoff}, KGE Runoff: {kge_runoff},Pbias Runoff: {pbias_runoff}\n")
                file.write(f"Name: {name}, Scenario: {scenario}, Epochs: {epochs}, ig, {ig}\n")
      
    
import io
with open(txtbook_path, 'a') as file:
    # Capture the summary
    summary_io = io.StringIO()
    model.summary(print_fn=lambda x: summary_io.write(x + "\n"))
    model_summary = summary_io.getvalue()
    file.write(model_summary+'\n')

F:/geodata/river_runoff_obs/note/note_2025-06-08_23-18-36_mon.txt
FGOALS-g3 256 16 0.3
256 16 0.3
ssp126
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 3)]               0         
                                                                 
 dense (Dense)               (None, 64)                256       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 attention_layer (AttentionL  (None, 32)               1056      
 ayer)                                                           
                                                                 
 dense_2 (Dense)             (None, 2)                 66        
                                                                 
Total params: 3,458
Tr

D:\Users\HP\miniconda3\envs\tensorflow-gpu\lib\site-packages\pandas\core\indexes\base.py:7588: FutureWarning: Dtype inference on a pandas object (Series, Index, ExtensionArray) is deprecated. The Index constructor will keep the original dtype in the future. Call `infer_objects` on the result to get the old behavior.
  return Index(sequences[0], name=names)


            Projected_Runoff  Projected_Evaporation
time                                               
2000-02-01          4.917418              16.991272
2000-03-01          5.366437              31.193680
2000-04-01          4.674500             114.634003
2000-05-01         11.900918             118.477684
2000-06-01         37.850609             213.865753
nse_runoff: 0.6308194718252531 nse_ep: 0.6930233610014395 kge_runoff: 0.6078194177652201 pbias_runoff: 7.934438820969187
[-0.04534025  0.5673368  -1.3377193 ]
ssp245
Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 3)]               0         
                                                                 
 dense_3 (Dense)             (None, 64)                256       
                                                                 
 dense_4 (Dense)             (None, 32)                20

D:\Users\HP\miniconda3\envs\tensorflow-gpu\lib\site-packages\pandas\core\indexes\base.py:7588: FutureWarning: Dtype inference on a pandas object (Series, Index, ExtensionArray) is deprecated. The Index constructor will keep the original dtype in the future. Call `infer_objects` on the result to get the old behavior.
  return Index(sequences[0], name=names)


7/7 [==============================] - 0s 14ms/step - loss: 1.0409 - val_loss: 0.8995
Epoch 2/256
7/7 [==============================] - 0s 5ms/step - loss: 1.0353 - val_loss: 0.8934
Epoch 3/256
7/7 [==============================] - 0s 5ms/step - loss: 1.0302 - val_loss: 0.8868
Epoch 4/256
7/7 [==============================] - 0s 6ms/step - loss: 1.0240 - val_loss: 0.8793
Epoch 5/256
7/7 [==============================] - 0s 5ms/step - loss: 1.0156 - val_loss: 0.8695
Epoch 6/256
7/7 [==============================] - 0s 6ms/step - loss: 1.0047 - val_loss: 0.8549
Epoch 7/256
7/7 [==============================] - 0s 5ms/step - loss: 0.9878 - val_loss: 0.8333
Epoch 8/256
7/7 [==============================] - 0s 5ms/step - loss: 0.9627 - val_loss: 0.7973
Epoch 9/256
7/7 [==============================] - 0s 5ms/step - loss: 0.9211 - val_loss: 0.7446
Epoch 10/256
7/7 [==============================] - 0s 6ms/step - loss: 0.8762 - val_loss: 0.6734
Epoch 11/256
7/7 [======================

...

D:\Users\HP\miniconda3\envs\tensorflow-gpu\lib\site-packages\pandas\core\indexes\base.py:7588: FutureWarning: Dtype inference on a pandas object (Series, Index, ExtensionArray) is deprecated. The Index constructor will keep the original dtype in the future. Call `infer_objects` on the result to get the old behavior.
  return Index(sequences[0], name=names)


[-0.23204164 -0.70145655 -0.23074289]
ssp370
Model: "model_194"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_195 (InputLayer)      [(None, 3)]               0         
                                                                 
 dense_582 (Dense)           (None, 64)                256       
                                                                 
 dense_583 (Dense)           (None, 32)                2080      
                                                                 
 attention_layer_194 (Attent  (None, 32)               1056      
 ionLayer)                                                       
                                                                 
 dense_584 (Dense)           (None, 2)                 66        
                                                                 
Total params: 3,458
Trainable params: 3,458
Non-trainable params: 0
__________

D:\Users\HP\miniconda3\envs\tensorflow-gpu\lib\site-packages\pandas\core\indexes\base.py:7588: FutureWarning: Dtype inference on a pandas object (Series, Index, ExtensionArray) is deprecated. The Index constructor will keep the original dtype in the future. Call `infer_objects` on the result to get the old behavior.
  return Index(sequences[0], name=names)


_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_196 (InputLayer)      [(None, 3)]               0         
                                                                 
 dense_585 (Dense)           (None, 64)                256       
                                                                 
 dense_586 (Dense)           (None, 32)                2080      
                                                                 
 attention_layer_195 (Attent  (None, 32)               1056      
 ionLayer)                                                       
                                                                 
 dense_587 (Dense)           (None, 2)                 66        
                                                                 
Total params: 3,458
Trainable params: 3,458
Non-trainable params: 0
_________________________________________________________________
Epoch 1/

D:\Users\HP\miniconda3\envs\tensorflow-gpu\lib\site-packages\pandas\core\indexes\base.py:7588: FutureWarning: Dtype inference on a pandas object (Series, Index, ExtensionArray) is deprecated. The Index constructor will keep the original dtype in the future. Call `infer_objects` on the result to get the old behavior.
  return Index(sequences[0], name=names)
